# Article 1 — figures and evidence for writing

This notebook is self-contained: it reads the closure CSVs and validated `OUTPUTS` rows, builds tables, and renders all figures in its own cells. It does not import `article1.paper` or display pre-rendered files.


In [ ]:
from itertools import product
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'docs/article1_closure').is_dir())
CLOSURE = ROOT / 'docs/article1_closure'
# Set VALIDATED explicitly to select another immutable analysis snapshot.
candidates = sorted((ROOT / 'OUTPUTS/article1_analysis').glob('*/tables/baseline_validated.csv'))
VALIDATED = candidates[-1] if candidates else None
DATASETS = ('mnist', 'fmnist', 'cifar')
REGIMES = ('iid', 'alpha1p0', 'alpha0p5', 'alpha0p1', 'multi', 'single')
REGIME_LABELS = ('IID', 'α=1', 'α=.5', 'α=.1', 'Multi', 'Single')
NAMES = {'mnist': 'MNIST', 'fmnist': 'Fashion-MNIST', 'cifar': 'CIFAR-10'}
METRICS = ('student_test_accuracy', 'student_test_nll')

manifest = json.loads((CLOSURE / 'manifest.json').read_text())
assert all(manifest['closure'][k] == 'cerrado' for k in ('masks', 'baseline', 'supervised', 'proxy_curve'))
main = pd.read_csv(CLOSURE / 'tables/main_contrast_summary.csv')
private = pd.read_csv(CLOSURE / 'tables/private_knowledge_summary.csv')
curve = pd.read_csv(CLOSURE / 'tables/proxy_curve_paired.csv')
validated = pd.read_csv(VALIDATED) if VALIDATED else pd.DataFrame()
if not validated.empty:
    assert validated.valid.eq(True).all(), 'Use audited valid rows only'
print('Per-seed snapshot:', VALIDATED or 'unavailable: absolute diagnostics skipped')
FIGURES = ROOT / 'OUTPUTS/article1_paper'
FIGURES.mkdir(parents=True, exist_ok=True)
for contrast in ('selection_logit', 'feddf_pooling', 'oracle_pooling', 'expertise_gain', 'oracle_expertise_gap', 'support'):
    for metric in METRICS:
        part = main[(main.contrast == contrast) & (main.metric == metric)]
        assert len(part) == 18 and set(zip(part.dataset, part.regime)) == set(product(DATASETS, REGIMES))
        assert part.n.eq(3).all() and part.seeds.eq('42,43,44').all()
assert len(curve) == 45
assert np.allclose(curve.student_test_accuracy_left - curve.student_test_accuracy_right, curve.delta_student_test_accuracy)
assert np.allclose(curve.student_test_nll_left - curve.student_test_nll_right, curve.delta_student_test_nll)


In [ ]:
def summary_figure(rows, panels, metric):
    scale = 100 if metric == "student_test_accuracy" else 1
    fig, axes = plt.subplots(
        len(panels), 3, figsize=(11, 2.8 * len(panels)), squeeze=False
    )
    for i, (contrast, title) in enumerate(panels.items()):
        for j, dataset in enumerate(DATASETS):
            part = rows[(rows.dataset == dataset) & (rows.metric == metric)]
            if contrast is not None:
                part = part[part.contrast == contrast]
            part = part.set_index("regime").loc[list(REGIMES)]
            ax = axes[i, j]
            ax.errorbar(
                range(6),
                part["mean"] * scale,
                yerr=part.sd * scale,
                fmt="o",
                capsize=3,
                color=("#2471A3", "#A04000", "#148F77")[j],
            )
            ax.axhline(0, color="0.5", lw=0.8)
            ax.set_xticks(range(6), REGIME_LABELS, rotation=25)
            ax.set_title(f"{NAMES[dataset]} | {title}", fontsize=10)
            ax.set_ylabel("Δ accuracy (pp)" if scale == 100 else "Δ NLL")
            ax.grid(axis="y", alpha=0.2)
            ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    plt.close(
        fig
    )  # <--- Evita la duplicación por el recolector automático de pyplot
    return fig

def paired_curve_figure(rows):
    fig, axes = plt.subplots(2, 3, figsize=(11, 6), squeeze=False)
    for j, regime in enumerate(('iid', 'alpha0p1', 'single')):
        part = rows[rows.regime == regime]
        for i, (metric, scale, ylabel) in enumerate((('delta_student_test_accuracy', 100, 'Δ accuracy (pp)'), ('delta_student_test_nll', 1, 'Δ NLL'))):
            ax = axes[i, j]
            for seed, seed_rows in part.groupby('seed'):
                seed_rows = seed_rows.sort_values('proxy_size'); ax.plot(seed_rows.proxy_size, seed_rows[metric] * scale, alpha=.45, lw=1, label=str(seed))
            stats = part.groupby('proxy_size')[metric].agg(['mean', 'std'])
            ax.errorbar(stats.index, stats['mean'] * scale, yerr=stats['std'] * scale, color='black', fmt='o-', capsize=3, label='Mean ± SD')
            ax.axhline(0, color='0.5', lw=.8); ax.set_xscale('log'); ax.set_xlabel('Proxy examples N'); ax.set_ylabel(ylabel); ax.set_title(f'CIFAR-10 | {regime}'); ax.spines[['top', 'right']].set_visible(False)
    axes[0, 0].legend(fontsize=8); fig.tight_layout(); return fig

def supervised_pairs(rows):
    cols = ['seed', 'proxy_size', 'student_test_accuracy_right', 'student_test_nll_right']
    ce = rows[cols].copy()
    for _, group in ce.groupby(['seed', 'proxy_size']):
        assert np.allclose(group[['student_test_accuracy_right', 'student_test_nll_right']], group[['student_test_accuracy_right', 'student_test_nll_right']].iloc[0])
    return ce.drop_duplicates(['seed', 'proxy_size']).sort_values(['seed', 'proxy_size'])

def expert_prob_pairs(rows):
    cols = ['regime', 'seed', 'proxy_size', 'student_test_accuracy_left', 'student_test_nll_left']
    expert = rows[rows.left == 'expert_prob'][cols].copy()
    return expert.sort_values(['regime', 'seed', 'proxy_size'])

def supervised_figure(ce, expert=None):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    metrics = (('student_test_accuracy_right', 'student_test_accuracy_left', 100, 'Test accuracy (%) ↑'), ('student_test_nll_right', 'student_test_nll_left', 1, 'Test NLL ↓'))
    for ax, (ce_metric, expert_metric, scale, ylabel) in zip(axes, metrics):
        methods = [('Supervised CE', ce, ce_metric, '#111111', ['seed']), ('EXPERT-prob', expert, expert_metric, '#0072B2', ['regime', 'seed'])]
        for label, data, metric, color, group_keys in methods:
            if data is None: continue
            for _, group in data.groupby(group_keys):
                group = group.sort_values('proxy_size'); ax.plot(group.proxy_size, group[metric] * scale, color=color, alpha=.25, lw=1)
            stats = data.groupby('proxy_size')[metric].agg(['mean', 'std'])
            ax.errorbar(stats.index, stats['mean'] * scale, yerr=stats['std'] * scale, color=color, fmt='o-', capsize=3, label=f'{label} mean ± SD')
        ax.set_xscale('log'); ax.set_xticks([100,500,1000,5000,10000], ['100','500','1000','5000','10000']); ax.tick_params(axis='x', labelrotation=25); ax.set_xlabel('Proxy examples N'); ax.set_ylabel(ylabel); ax.set_title('CIFAR-10 | supervised vs EXPERT-prob'); ax.spines[['top', 'right']].set_visible(False)
    axes[0].legend(fontsize=8); fig.tight_layout(); return fig


## 0. Student performance, including supervision

These are selected reference arms, not variants chosen as universally best. Pooling effects are evaluated separately below. Confidence, Consensus and Energy have incomplete coverage; missing results remain gaps. The optional absolute plot requires audited per-seed results; published contrasts remain available without them.


In [ ]:
overview_table = pd.DataFrame()
if validated.empty:
    print('Absolute overview requires baseline_validated.csv from your analysis snapshot.')
else:
    OVERVIEW = ('feddf_prob', 'oracle_logit', 'expert_prob', 'confidence_logit', 'consensus_logit', 'energy_logit')
    assert set(OVERVIEW).issubset(validated.method)
    assert not validated[validated.method.isin(OVERVIEW)].duplicated(['dataset', 'regime', 'seed', 'method']).any()
    overview_table = (validated[validated.method.isin(OVERVIEW)].melt(id_vars=['dataset','regime','seed','method'], value_vars=list(METRICS), var_name='metric', value_name='seed_value').groupby(['dataset','regime','method','metric'], as_index=False).agg(mean=('seed_value','mean'), n=('seed','nunique')))
    ce = private[private.metric.isin(['student_test_accuracy_right','student_test_nll_right'])].copy()
    ce['metric'] = ce.metric.map({'student_test_accuracy_right':'student_test_accuracy', 'student_test_nll_right':'student_test_nll'}); ce['method'] = 'supervised_proxy_ce'
    overview_table = pd.concat([overview_table, ce[['dataset','regime','method','metric','mean','n']]], ignore_index=True)
    style = {'feddf_prob':('FedDF-prob','#7f7f7f'), 'oracle_logit':('ORACLE-logit','#9467bd'), 'expert_prob':('EXPERT-prob','#0072B2'), 'confidence_logit':('Confidence','#CC79A7'), 'consensus_logit':('Consensus','#009E73'), 'energy_logit':('Energy','#E69F00'), 'supervised_proxy_ce':('CE (N=10000)','#111111')}
    fig_overview, axes = plt.subplots(2, 3, figsize=(12,7), squeeze=False)
    for j, dataset in enumerate(DATASETS):
        for i, (metric, scale, ylabel) in enumerate((('student_test_accuracy',100,'Test accuracy (%) ↑'), ('student_test_nll',1,'Test NLL ↓'))):
            ax = axes[i,j]
            for method, (label, color) in style.items():
                part = overview_table[(overview_table.dataset == dataset) & (overview_table.metric == metric) & (overview_table.method == method)].set_index('regime').reindex(REGIMES)
                ax.plot(range(6), part['mean'] * scale, label=label, color=color, linestyle=':' if method == 'supervised_proxy_ce' else '-', marker=None if method == 'supervised_proxy_ce' else 'o', markersize=3)
            ax.set_title(NAMES[dataset]); ax.set_ylabel(ylabel); ax.set_xticks(range(6), REGIME_LABELS, rotation=25); ax.grid(axis='y', alpha=.2); ax.spines[['top','right']].set_visible(False)
            if i == 0: ax.set_ylim(0,100)
            else: ax.set_ylim(bottom=0)
    handles, labels = axes[0,0].get_legend_handles_labels(); fig_overview.legend(handles, labels, loc='upper center', ncol=4, fontsize=9); fig_overview.tight_layout(rect=(0,0,1,.9))
    fig_overview


In [ ]:
display(overview_table)


## 1. Who contributes?


In [ ]:
routing = {'selection_logit':'ORACLE − FedDF (logit)', 'expertise_gain':'EXPERT − FedDF (prob)'}
summary_figure(main, routing, 'student_test_accuracy')


In [ ]:
summary_figure(main, routing, 'student_test_nll')


## 2. Target vs Student: full support and expertise-restricted support

For proxy sample $x$, both arms select $S(x)=\{k:M_{k,y(x)}=1\}$ and use the same temperature $T=8$. With $p_k=\operatorname{softmax}(z_k/T)$, full EXPERT averages complete vectors. SR instead averages
$$p^{SR}_{k,c}=\frac{M_{k,c}p_{k,c}}{\sum_j M_{k,j}p_{k,j}}.$$
Here **support means the expertise mask**, not the classes present in train. Even an IID teacher can fail the expertise threshold. Empty selections use the same FedDF-logit fallback in both arms.

Every difference below is **SR − full EXPERT**. Error bars are sample SD across three paired seeds, not confidence intervals. Target metrics use the training proxy at T=8; student metrics use the official test at T=1. Their absolute levels are not generalization gaps.

Because every selected teacher supports the true proxy class, renormalization cannot reduce its probability of that class. Target NLL therefore cannot increase (up to numerical precision); this is partly a consequence of using proxy labels for routing. Target accuracy and student accuracy have no corresponding monotonic guarantee.


In [ ]:
# Published, paired contrasts: always SR minus full EXPERT.
support = main[main.contrast.eq('support')].copy()
support_metrics = [
    ('target_accuracy', 100, 'Δ target accuracy (proxy, pp)'),
    ('target_nll', 1, 'Δ target NLL (proxy, T=8)'),
    ('target_entropy', 1, 'Δ target entropy (proxy, T=8)'),
    ('student_test_accuracy', 100, 'Δ student accuracy (test, pp)'),
    ('student_test_nll', 1, 'Δ student NLL (test, T=1)'),
]
assert not support.duplicated(['dataset','regime','metric']).any()
assert len(support) == 90 and support.n.eq(3).all() and support.seeds.eq('42,43,44').all()
fig_target_student, axes = plt.subplots(5,3,figsize=(12,13),squeeze=False)
for i, (metric, scale, label) in enumerate(support_metrics):
    for j, dataset in enumerate(DATASETS):
        part = support[(support.metric==metric)&(support.dataset==dataset)].set_index('regime').reindex(REGIMES)
        ax = axes[i,j]
        ax.errorbar(range(6), scale*part['mean'], yerr=scale*part.sd, fmt='o', capsize=3)
        ax.axhline(0,color='.5',linewidth=.8)
        ax.set_xticks(range(6),REGIME_LABELS,rotation=35)
        if i==0: ax.set_title(NAMES[dataset])
        if j==0: ax.set_ylabel(label)
        ax.spines[['top','right']].set_visible(False)
fig_target_student.suptitle('Support restriction: SR − full EXPERT; paired mean ± SD')
fig_target_student.tight_layout()
for extension in ('png','pdf'):
    fig_target_student.savefig(FIGURES/f'target_vs_student.{extension}',dpi=180,bbox_inches='tight')
display(support.pivot(index=['dataset','regime'],columns='metric',values='mean'))


In [ ]:
# Absolute target/student metrics and ACTUAL pre-restriction outside mass.
# This metric averages selected teacher/sample events; fallback events are excluded.
support_pairs = pd.DataFrame()
if validated.empty:
    print('Actual outside mass and absolute metrics need the audited per-seed snapshot; no estimates substituted.')
else:
    keys = ['dataset','regime','seed']
    arms = validated[validated.method.isin(['expert_prob','expert_prob_sr'])].copy()
    assert not arms.duplicated(keys+['method']).any()
    full = arms[arms.method.eq('expert_prob')]
    sr = arms[arms.method.eq('expert_prob_sr')]
    support_pairs = full.merge(sr,on=keys,suffixes=('_full','_sr'),validate='one_to_one')
    assert len(support_pairs)==54, 'Need all 54 paired conditions'
    assert set(map(tuple,support_pairs[keys].to_numpy())) == set(product(DATASETS,REGIMES,(42,43,44)))
    assert support_pairs.target_revision_sr.eq(2).all()
    assert support_pairs.temperature_full.eq(8).all() and support_pairs.proxy_size_full.eq(10000).all()
    shared = ['cache_sha256','M_sha256','proxy_sha256','proxy_labels_sha256',
              'student_init_sha256','consumed_batches_sha256','updates','temperature',
              'training_recipe_json','fallback_count','mean_selected_teachers']
    for field in shared:
        assert support_pairs[field+'_full'].eq(support_pairs[field+'_sr']).all(), field
    mass = 'pre_restriction_outside_support_mass'
    a,b = support_pairs[mass+'_full'],support_pairs[mass+'_sr']
    assert np.allclose(a,b,equal_nan=True), 'Both arms must measure the SAME pre-restriction mass'
    assert a.dropna().between(-1e-12,1+1e-12).all()
    for metric,_,_ in support_metrics:
        support_pairs['delta_'+metric] = support_pairs[metric+'_sr']-support_pairs[metric+'_full']
        observed = support_pairs.groupby(['dataset','regime'])['delta_'+metric].mean().sort_index()
        recorded = support[support.metric.eq(metric)].set_index(['dataset','regime'])['mean'].sort_index()
        assert np.allclose(observed,recorded), 'Snapshot and published closure differ: choose a matching snapshot'
    assert (support_pairs.delta_target_nll <= 1e-6).all()
    absolute_columns = [m+suffix for m,_,_ in support_metrics for suffix in ('_full','_sr')]
    display(support_pairs.groupby(['dataset','regime'])[absolute_columns+[mass+'_full']].mean())
    fig_mass, axes = plt.subplots(2,3,figsize=(12,7))
    for j,dataset in enumerate(DATASETS):
        part = support_pairs[support_pairs.dataset.eq(dataset)]
        grouped = part.groupby('regime')[mass+'_full'].agg(['mean','std','count']).reindex(REGIMES)
        axes[0,j].errorbar(range(6),100*grouped['mean'],yerr=100*grouped['std'],fmt='o',capsize=3)
        axes[0,j].set_xticks(range(6),REGIME_LABELS,rotation=35)
        axes[0,j].set_ylim(0,100); axes[0,j].set_title(NAMES[dataset])
        for regime,label in zip(REGIMES,REGIME_LABELS):
            sub = part[part.regime.eq(regime)]
            axes[1,j].scatter(100*sub[mass+'_full'],100*sub.delta_student_test_accuracy,label=label)
        axes[1,j].axhline(0,color='.5',linewidth=.8)
        axes[1,j].set_xlabel('Observed outside-expertise mass (%)')
    axes[0,0].set_ylabel('Outside-expertise mass (%)')
    axes[1,0].set_ylabel('Δ student accuracy: SR − full (pp)')
    axes[1,2].legend(fontsize=8)
    fig_mass.suptitle('Actual mass before restriction; association is not causal evidence')
    fig_mass.tight_layout()
    for extension in ('png','pdf'):
        fig_mass.savefig(FIGURES/f'outside_expertise_mass.{extension}',dpi=180,bbox_inches='tight')
    support_pairs.to_csv(FIGURES/'target_student_seed_pairs.csv',index=False)


### What this establishes—and what remains a hypothesis

In the published CIFAR-10 contrasts, SR lowers target NLL in all six regimes, lowers student accuracy in five of six, and increases student test NLL in all six. `single` is the accuracy exception. These observations do not support the earlier claim of negligible IID effects.

Outside-mask mass is not automatically useful dark knowledge. SR simultaneously removes inter-class probabilities and changes concentration. The ablation measures their combined effect; it does not isolate a causal mechanism or establish that this mass estimates rejection. The scatter is descriptive, with shared teachers and regimes—not 54 independent causal replications.

If a selected teacher has exactly one supported class, that class must be the proxy label and its SR vector is one-hot. Under labeled routing this is not an arbitrary wrong-class pseudo-label. An entropy-matched full-support control would help distinguish concentration from information removal; it has not been run.


## Pooling: probability versus logit averages


In [ ]:
pooling = {'feddf_pooling':'FedDF: prob − logit', 'oracle_pooling':'ORACLE: prob − logit'}
summary_figure(main, pooling, 'student_test_accuracy')

In [ ]:
summary_figure(main, pooling, 'student_test_nll')


## Presence versus measured competence (new control, pending execution)

`presence_prob` uses $A_{k,c}=1[n^{train}_{k,c}>0]$ and selects teachers with $A_{k,y(x)}=1$. It averages **full** probability vectors, exactly like `expert_prob`; only the mask definition changes. It does not use validation, expertise, proxy or test counts to construct A. With full class presence it reduces to FedDF-prob.

Compare EXPERT − presence, paired by dataset/regime/seed, with the same teachers, proxy, initialization, batches and budget. Report accuracy/NLL, coverage and fallback changes. This measures the incremental utility of the accuracy-based mask over a binary training-presence rule. It does not isolate mask quality from changes in the number of selected teachers and does not establish superiority to every possible presence rule. No result is asserted before this control finishes.


In [ ]:
presence_file = ROOT/'OUTPUTS/article1_v3/results_presence.csv'
if presence_file.exists() and not validated.empty:
    from article1.presence import compare_presence
    presence_pairs = compare_presence(pd.read_csv(presence_file), validated)
    print(f'Presence control: {len(presence_pairs)}/54 paired conditions; partial results are provisional')
    metrics = ['delta_student_test_accuracy','delta_student_test_nll',
               'delta_fallback_rate','delta_mean_selected_teachers']
    display(presence_pairs.groupby(['dataset','regime'])[metrics].agg(['mean','std','count']))
    fig_presence, axes = plt.subplots(2,3,figsize=(12,7))
    for i,(metric,scale,label) in enumerate([
        ('delta_student_test_accuracy',100,'EXPERT − presence accuracy (pp)'),
        ('delta_student_test_nll',1,'EXPERT − presence NLL')]):
        for j,dataset in enumerate(DATASETS):
            part = presence_pairs[presence_pairs.dataset.eq(dataset)]
            summary = part.groupby('regime')[metric].agg(['mean','std','count']).reindex(REGIMES)
            ax = axes[i,j]
            ax.errorbar(range(6),scale*summary['mean'],yerr=scale*summary['std'],fmt='o',capsize=3)
            ax.axhline(0,color='.5',linewidth=.8)
            ax.set_xticks(range(6),REGIME_LABELS,rotation=35)
            if i==0: ax.set_title(NAMES[dataset])
            if j==0: ax.set_ylabel(label)
    fig_presence.suptitle('Incremental utility of measured competence; paired mean ± SD')
    fig_presence.tight_layout()
    for extension in ('png','pdf'):
        fig_presence.savefig(FIGURES/f'presence_control.{extension}',dpi=180,bbox_inches='tight')
    presence_pairs.to_csv(FIGURES/'presence_seed_pairs.csv',index=False)
else:
    print('Pending: python run_article1_pipeline.py --phase presence --execute --device cuda')


## 3. What does KD add beyond public labels?

EXPERT-prob − CE at N=10000.


In [ ]:
private_effects = private.copy(); private_effects['metric'] = private_effects.metric.str.removeprefix('delta_')
summary_figure(private_effects, {None:'EXPERT-prob − CE'}, 'student_test_accuracy')


In [ ]:
summary_figure(private_effects, {None:'EXPERT-prob − CE'}, 'student_test_nll')


## 3b. Supervised learning curve, alone


In [ ]:
ce_pairs = supervised_pairs(curve)
expert_pairs = expert_prob_pairs(curve)
supervised_figure(ce_pairs, expert_pairs)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.8, 3.8))
regime_style = {
    'iid': ('IID', '#0072B2', 'o'),
    'alpha0p1': ('α=.1', '#009E73', 's'),
    'single': ('Single', '#D55E00', '^'),
}
metrics = (
    ('student_test_accuracy_right', 'student_test_accuracy_left', 100, 'Test accuracy (%) ↑'),
    ('student_test_nll_right', 'student_test_nll_left', 1, 'Test NLL ↓'),
)

for ax, (ce_metric, expert_metric, scale, ylabel) in zip(axes, metrics):
    # Supervised CE
    for _, group in ce_pairs.groupby('seed'):
        group = group.sort_values('proxy_size')
        ax.plot(group.proxy_size, group[ce_metric] * scale, color='0.35', alpha=.18, lw=1)
    ce_stats = ce_pairs.groupby('proxy_size')[ce_metric].agg(['mean', 'std'])
    ax.errorbar(
        ce_stats.index,
        ce_stats['mean'] * scale,
        yerr=ce_stats['std'] * scale,
        color='#111111',
        fmt='o-',
        capsize=3,
        label='Supervised CE mean ± SD'
    )

    # EXPERT-prob: media ± SD por régimen de heterogeneidad
    for regime, (label, color, marker) in regime_style.items():
        part = expert_pairs[expert_pairs.regime == regime]
        stats = part.groupby('proxy_size')[expert_metric].agg(['mean', 'std'])
        ax.errorbar(
            stats.index,
            stats['mean'] * scale,
            yerr=stats['std'] * scale,
            color=color,
            fmt=f'{marker}-',
            capsize=3,
            markersize=4,
            lw=1.3,
            label=f'EXPERT-prob {label}'
        )

    ax.set_xscale('log')
    ax.set_xticks([100, 500, 1000, 5000, 10000], ['100', '500', '1000', '5000', '10000'])
    ax.tick_params(axis='x', labelrotation=25)
    ax.set_xlabel('Proxy examples N')
    ax.set_ylabel(ylabel)
    ax.set_title('CIFAR-10 | heterogeneity split')
    ax.spines[['top', 'right']].set_visible(False)

axes[0].legend(fontsize=8, ncol=2)
fig.tight_layout()

In [ ]:
ce_pairs

In [ ]:
expert_pairs

## 4. How does the advantage depend on N?


In [ ]:
paired_curve_figure(curve);


In [ ]:
curve
